# Device and distributed execution

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SilenceX12138/TabFORGE/blob/master/docs/tutorial/notebooks/generation/03_device_and_distributed_execution.ipynb)

> **Colab setup:** Open this notebook with the badge, then follow the [tutorial setup guide](https://silencex12138.github.io/TabFORGE/#/tutorials.md) to install `tabforge-ml[tutorials]` from PyPI in this runtime. Select a GPU for pretrained workloads.

## 1. Overview

This tutorial selects CPU or CUDA through one runtime configuration and explains how the same
standalone estimator participates in four-GPU `torchrun` execution.

In [ ]:
%pip install "tabforge-ml[tutorials]"

## 2. Imports and setup

`DEVICE=auto` selects CUDA when available.

In [ ]:
import os
import tempfile
from pathlib import Path

from tabforge import (
    MODEL_ID,
    TabFORGEEmbeddingConfig,
    TabFORGERuntimeConfig,
    TabFORGETrainingConfig,
)

RANDOM_STATE = 7
MAX_TRAIN_ROWS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_ROWS", "256"))
MAX_STEPS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_STEPS", "10"))
DEVICE = os.environ.get("TABFORGE_TUTORIAL_DEVICE", "auto")

assert MODEL_ID == "tabforge-v1"

## 3. Prepare a real-world dataset

`TabularDataset` downloads German Credit from OpenML, keeps the mixed numerical/categorical schema,
and performs a reproducible stratified split. The tutorial caps the training rows to keep execution
short; increase `TABFORGE_TUTORIAL_MAX_ROWS` for a larger run.

In [ ]:
from tabcamel.data.dataset import TabularDataset

dataset = TabularDataset("credit-g", task_type="classification")
split = dataset.split("stratified", train_size=0.8, random_state=RANDOM_STATE)
train_set = split["train_set"]
test_set = split["test_set"]

X_train = train_set.X_df.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
y_train = train_set.y_s.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
X_test = test_set.X_df.reset_index(drop=True)
y_test = test_set.y_s.reset_index(drop=True)
categorical_features = tuple(dataset.categorical_feature_list)

print({"train": X_train.shape, "test": X_test.shape, "target": y_train.name})

## 4. Configure TabFORGE

Single-process notebooks use `strategy="auto"`; `torchrun` workers use `strategy="ddp"`.

In [ ]:
from tabforge import TabFORGEGenerator

runtime_config = TabFORGERuntimeConfig(device=DEVICE, strategy="auto")
model = TabFORGEGenerator(
    task="classification",
    categorical_features=categorical_features,
    embedding_config=TabFORGEEmbeddingConfig(model_path="auto", n_folds=2),
    training_config=TabFORGETrainingConfig(max_steps=MAX_STEPS, batch_size=128),
    runtime_config=runtime_config,
    random_state=RANDOM_STATE,
)

## 5. Fit

The progress bar and W&B training trace are emitted only by rank zero.

In [ ]:
model.fit(X_train, y_train, checkpoint="pretrained")

## 6. Generate

Distributed generation shards the requested count and gathers rows in rank order.

In [ ]:
generated = model.generate(16, random_state=RANDOM_STATE + 1)
print({"configured_device": DEVICE, "generated_rows": len(generated[0])})

## 7. Launch on four GPUs

First export the complete training split with `train_set.X_df.to_csv("train.csv", index=False)` in the notebook. The CLI uses ten embedding folds; retain the complete split for pretrained distributed runs, even when the notebook uses a smaller training-row cap. For a small CLI smoke test, add `--model-path mock --embedding-dimension 8` to `fit`. From the directory containing that CSV, run the CLI from a terminal so `torchrun` can create one process per GPU. TabFORGE owns embedding,
training, and inference distribution; rank zero writes the checkpoint and output.

```bash
CUDA_VISIBLE_DEVICES=0,1,2,3 torchrun \
  --standalone \
  --nproc-per-node=4 \
  -m tabforge.cli fit \
  --input train.csv \
  --checkpoint model \
  --estimator generator \
  --task unsupervision \
  --strategy ddp

CUDA_VISIBLE_DEVICES=0,1,2,3 torchrun \
  --standalone \
  --nproc-per-node=4 \
  -m tabforge.cli generate \
  --checkpoint model \
  --n-samples 1000 \
  --output synthetic.csv
```

## 8. Save and load

Checkpoint writes are rank-zero only under distributed execution.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = model.save_checkpoint(Path(directory))
    restored = TabFORGEGenerator.restore_from_checkpoint(checkpoint_path, device=DEVICE)

print({"checkpoint": checkpoint_path.name, "restored_type": type(restored).__name__})

## 9. Optional W&B logging

Set `log_wandb=True`, a project, and `wandb_dir` in `TabFORGERuntimeConfig`. In DDP, rank zero owns the
run while all workers participate in computation.